# ODI Session 85 - SQL Extraction from SNP_SESS_TASK_LOG

This notebook connects to the Oracle ODI Work Repository, queries `SNP_SESS_TASK_LOG` for **Session 85**, and generates a `.txt` file in the same format as the existing ODI Session Log files.

**Output format reference:** `RX_Demo/ODI_Session_log/` txt files

---

## Format of Output TXT File
Each task is written as:
```
SCEN_TASK_NO in {N}
<SQL text from TASK_TXT1 + TASK_TXT2>
```
Tasks with no SQL text will have just the `SCEN_TASK_NO in {N}` line.

---
## Step 1: Install Oracle cx_Oracle Driver (if not installed)

In [ ]:
# Install cx_Oracle if not already available
# Uncomment the line below if running for the first time
# !pip install cx_Oracle

import cx_Oracle
import os
import re

print("cx_Oracle version:", cx_Oracle.version)

---
## Step 2: Configure Oracle ODI Repository Connection

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Oracle ODI Work Repository connection parameters
# Update these values to match your environment
# ─────────────────────────────────────────────────────────────────

ODI_HOST     = 'your_oracle_host'       # e.g. 'localhost' or '10.0.0.1'
ODI_PORT     = 1521                     # Default Oracle listener port
ODI_SERVICE  = 'your_service_name'      # e.g. 'ORCL' or 'BIAPPS'
ODI_USER     = 'your_odi_repo_user'     # ODI Work Repository schema user
ODI_PASSWORD = 'your_odi_repo_password' # ODI Work Repository schema password

# Session number to extract
SESSION_NO = 85

# Output file path
OUTPUT_DIR  = os.path.dirname(os.path.abspath('__file__'))  # same dir as this notebook
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f'Session_{SESSION_NO}.txt')

print(f"Will extract Session No : {SESSION_NO}")
print(f"Output file             : {OUTPUT_FILE}")

---
## Step 3: Connect to Oracle ODI Work Repository

In [ ]:
# Build DSN and connect
dsn = cx_Oracle.makedsn(
    host    = ODI_HOST,
    port    = ODI_PORT,
    service_name = ODI_SERVICE
)

conn = cx_Oracle.connect(
    user     = ODI_USER,
    password = ODI_PASSWORD,
    dsn      = dsn
)

print(f"Connected to Oracle: {conn.version}")
print(f"DSN: {ODI_HOST}:{ODI_PORT}/{ODI_SERVICE}")

---
## Step 4: Query SNP_SESS_TASK_LOG for Session 85

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Query SNP_SESS_TASK_LOG for session 85
#
# Key columns:
#   SESS_NO       - Session identifier
#   SCEN_TASK_NO  - Task sequence number within the scenario
#   TASK_TXT1     - First segment of SQL text (up to 4000 chars)
#   TASK_TXT2     - Second segment of SQL text (continuation)
#   TASK_TXT3     - Third segment of SQL text (if needed)
#   TASK_TXT4     - Fourth segment of SQL text (if needed)
# ─────────────────────────────────────────────────────────────────

sql_query = """
SELECT
    SCEN_TASK_NO,
    NVL(TASK_TXT1, '') || NVL(TASK_TXT2, '') || NVL(TASK_TXT3, '') || NVL(TASK_TXT4, '') AS TASK_SQL
FROM
    SNP_SESS_TASK_LOG
WHERE
    SESS_NO = :session_no
ORDER BY
    SCEN_TASK_NO ASC
"""

cursor = conn.cursor()
cursor.execute(sql_query, session_no=SESSION_NO)

rows = cursor.fetchall()
cursor.close()

print(f"Total tasks found for Session {SESSION_NO}: {len(rows)}")
print()

# Preview first few rows
for i, (scen_task_no, task_sql) in enumerate(rows[:5]):
    preview = (task_sql or '')[:80].replace('\n', ' ')
    print(f"  SCEN_TASK_NO={scen_task_no:>4}  SQL preview: {preview}")

---
## Step 5: Format and Write the TXT File

Format matches existing ODI Session Log files in `RX_Demo/ODI_Session_log/`:
```
SCEN_TASK_NO in {N}
<SQL text>
SCEN_TASK_NO in {N+10}
<SQL text>
...
```

In [ ]:
def format_session_txt(rows):
    """
    Format session task rows into the ODI Session Log txt format.
    
    Format:
        SCEN_TASK_NO in {N}
        <SQL text>  (omitted if empty)
        SCEN_TASK_NO in {N+10}
        ...
    """
    lines = []

    for scen_task_no, task_sql in rows:
        # Write the task number header
        lines.append(f"SCEN_TASK_NO in {{{scen_task_no}}}")

        # Write SQL text if present (strip trailing whitespace only)
        if task_sql and task_sql.strip():
            lines.append(task_sql.rstrip())
            lines.append("  ")   # trailing spaces after SQL block (matches reference format)

    # Trailing blank lines (matches reference format)
    lines.append("")
    lines.append("")
    lines.append("")

    return "\n".join(lines)


# Generate formatted content
txt_content = format_session_txt(rows)

# Write to output file
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write(txt_content)

print(f"TXT file written successfully: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE):,} bytes")

# Preview first 30 lines
print("\n--- Preview (first 30 lines) ---")
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 30:
            print("  ... (truncated)")
            break
        print(f"  {i+1:>4}: {line}", end='')

---
## Step 6: Close Connection

In [ ]:
conn.close()
print("Oracle connection closed.")
print()
print(f"Session {SESSION_NO} TXT file saved to:")
print(f"  {OUTPUT_FILE}")

---
## Reference: SNP_SESS_TASK_LOG SQL Query

Run the following SQL directly in SQL*Plus / SQL Developer to view Session 85 tasks:

```sql
SELECT
    SCEN_TASK_NO,
    NVL(TASK_TXT1, '') || NVL(TASK_TXT2, '') || NVL(TASK_TXT3, '') || NVL(TASK_TXT4, '') AS TASK_SQL
FROM
    SNP_SESS_TASK_LOG
WHERE
    SESS_NO = 85
ORDER BY
    SCEN_TASK_NO ASC;
```

### Output TXT Format
Each task block follows this pattern:
```
SCEN_TASK_NO in {<task_number>}
<SQL text from TASK_TXT1+TASK_TXT2+TASK_TXT3+TASK_TXT4>
```

### Notes
- Tasks with **no SQL** (variable tasks, placeholder steps) produce only the `SCEN_TASK_NO in {N}` line.
- The `SNP_SESS_TASK_LOG` table may split long SQL across `TASK_TXT1`, `TASK_TXT2`, `TASK_TXT3`, `TASK_TXT4` — all are concatenated.
- Task numbers increment by 10 for standard steps.